In [ ]:
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2
import numpy as np
import PIL
import io
import math


In [ ]:
def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(jpg_as_np, flags=1)
    return img

def bbox_to_bytes(bbox_array):
    bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA')
    iobuf = io.BytesIO()
    bbox_PIL.save(iobuf, format='png')
    bbox_bytes = 'data:image/png;base64,{}'.format(
        str(b64encode(iobuf.getvalue()), 'utf-8'))
    return bbox_bytes

def video_stream():
    js = Javascript('''
        var video;
        var div = null;
        var stream;
        var captureCanvas;
        var imgElement;
        var labelElement;
        var pendingResolve = null;
        var shutdown = false;

        function removeDom() {
            stream.getVideoTracks()[0].stop();
            video.remove();
            div.remove();
            video = null; div = null; stream = null;
            imgElement = null; captureCanvas = null; labelElement = null;
        }

        function onAnimationFrame() {
            if (!shutdown) window.requestAnimationFrame(onAnimationFrame);
            if (pendingResolve) {
                var result = "";
                if (!shutdown) {
                    captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
                    result = captureCanvas.toDataURL('image/jpeg', 0.8);
                }
                var lp = pendingResolve;
                pendingResolve = null;
                lp(result);
            }
        }
        async function createDom() {
            if (div !== null) return stream;

            div = document.createElement('div');
            div.style.border = '2px solid black';
            div.style.padding = '3px';
            div.style.width = '100%';
            div.style.maxWidth = '600px';
            document.body.appendChild(div);

            const modelOut = document.createElement('div');
            modelOut.innerHTML = "<span>Status: </span>";
            labelElement = document.createElement('span');
            labelElement.innerText = 'No data';
            labelElement.style.fontWeight = 'bold';
            modelOut.appendChild(labelElement);
            div.appendChild(modelOut);

            video = document.createElement('video');
            video.style.display = 'block';
            video.width = div.clientWidth - 6;
            video.setAttribute('playsinline', '');
            video.onclick = () => { shutdown = true; };
            stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: "environment"}});
            div.appendChild(video);

            imgElement = document.createElement('img');
            imgElement.style.position = 'absolute';
            imgElement.style.zIndex = 1;
            imgElement.onclick = () => { shutdown = true; };
            div.appendChild(imgElement);

            const instruction = document.createElement('div');
            instruction.innerHTML =
                '<span style="color: red; font-weight: bold;">' +
                'Click here or on the video to stop</span>';
            div.appendChild(instruction);
            instruction.onclick = () => { shutdown = true; };

            video.srcObject = stream;
            await video.play();

            captureCanvas = document.createElement('canvas');
            captureCanvas.width = 640;
            captureCanvas.height = 480;
            window.requestAnimationFrame(onAnimationFrame);
            return stream;
        }

        async function stream_frame(label, imgData) {
            if (shutdown) { removeDom(); shutdown = false; return ''; }

            var preCreate = Date.now();
            stream = await createDom();

            var preShow = Date.now();
            if (label != "") labelElement.innerHTML = label;

            if (imgData != "") {
                var videoRect = video.getClientRects()[0];
                imgElement.style.top = videoRect.top    + "px";
                imgElement.style.left = videoRect.left   + "px";
                imgElement.style.width = videoRect.width  + "px";
                imgElement.style.height = videoRect.height + "px";
                imgElement.src = imgData;
            }

            var preCapture = Date.now();
            var result = await new Promise(function(resolve, reject) {
                pendingResolve = resolve;
            });
            shutdown = false;

            return {'create':  preShow   - preCreate,
                    'show':    preCapture - preShow,
                    'capture': Date.now() - preCapture,
                    'img':     result};
        }
    ''')
    display(js)

def video_frame(label, bbox):
    """Fetch one frame from the JS stream, send back the overlay."""
    data = eval_js('stream_frame("{}", "{}")'.format(label, bbox))
    return data

In [ ]:
def hough_circles_acc(edge_img, r_min, r_max):
    height, width = edge_img.shape
    num_r = r_max - r_min + 1
    accumulator = np.zeros((height, width, num_r), dtype=np.uint64)

    edge_points = np.argwhere(edge_img > 0)
    thetas = np.deg2rad(np.arange(0, 360))
    cos_t = np.cos(thetas)
    sin_t = np.sin(thetas)

    for y, x in edge_points:
        for r_index, r in enumerate(range(r_min, r_max + 1)):
            xc_vals = np.round(x - r * cos_t).astype(int)
            yc_vals = np.round(y - r * sin_t).astype(int)

            valid = (xc_vals >= 0) & (xc_vals < width) & \
                    (yc_vals >= 0) & (yc_vals < height)

            np.add.at(accumulator, (yc_vals[valid], xc_vals[valid], r_index), 1)

    return accumulator


def hough_circles_peaks(accumulator, r_min, t, s, min_dist=50):
    circles = []
    yc_idx, xc_idx, r_idx = np.where(accumulator >= t)
    if len(yc_idx) == 0:
        return circles

    votes = accumulator[yc_idx, xc_idx, r_idx]
    sorted_indices = np.argsort(votes)[::-1]

    for i in sorted_indices:
        yc = int(yc_idx[i])
        xc = int(xc_idx[i])
        r = int(r_idx[i]) + r_min
        v = int(votes[i])

        too_close = any(
            math.hypot(xc - ex, yc - ey) < min_dist
            for (ex, ey, er, _) in circles
        )
        if not too_close:
            circles.append((xc, yc, r, v))
        if len(circles) >= s:
            break

    return circles


def draw_circles_overlay(circles, overlay_shape=(480, 640)):
    overlay = np.zeros([overlay_shape[0], overlay_shape[1], 4], dtype=np.uint8)

    for (x, y, r, v) in circles:
        cv2.circle(overlay, (x, y), r, (0, 255, 0, 255), 2)
        cv2.circle(overlay, (x, y), 2, (0, 0, 255, 255), 3)
        cv2.putText(overlay, f"r={r}", (x - 15, y - r - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 0, 255), 1)

    overlay[:, :, 3] = (overlay.max(axis=2) > 0).astype(np.uint8) * 255
    return overlay

In [ ]:
R_MIN = 20
R_MAX = 60
CANNY_LOW = 50
CANNY_HIGH = 150
THRESHOLD_PCT = 0.35
MAX_CIRCLES = 5
MIN_DIST = 40
SCALE = 0.5

video_stream()

label_html = 'Detecting circles...'
bbox = ''
frame_idx = 0

while True:
    js_reply = video_frame(label_html, bbox)
    if not js_reply:
        break

    # 1. Decode webcam frame
    img = js_to_image(js_reply["img"])

    small = cv2.resize(img, (0, 0), fx=SCALE, fy=SCALE)
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (9, 9), 2)
    canny = cv2.Canny(blurred, CANNY_LOW, CANNY_HIGH)

    r_min_s = max(1, int(R_MIN * SCALE))
    r_max_s = max(r_min_s + 1, int(R_MAX * SCALE))

    acc = hough_circles_acc(canny, r_min_s, r_max_s)
    t = int(THRESHOLD_PCT * 360)
    circles = hough_circles_peaks(acc, r_min_s, t, s=MAX_CIRCLES, min_dist=int(MIN_DIST * SCALE))

    circles_full = [
        (int(x / SCALE), int(y / SCALE), int(r / SCALE), v)
        for (x, y, r, v) in circles
    ]

    overlay = draw_circles_overlay(circles_full, overlay_shape=(480, 640))
    bbox_bytes = bbox_to_bytes(overlay)
    bbox = bbox_bytes

    label_html = f'Frame {frame_idx} | Circles found: {len(circles_full)} | r=[{R_MIN},{R_MAX}] t={t}'
    frame_idx += 1

<IPython.core.display.Javascript object>

/tmp/ipykernel_428/1168788723.py:8: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  bbox_PIL  = PIL.Image.fromarray(bbox_array, 'RGBA')


KeyboardInterrupt: 